<a href="https://colab.research.google.com/github/varshini-wilson8/UltraSound_Nerve_Segmentation/blob/main/Copy_of_ULTRASUOND_NERVE_SEGMENTATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

print("Setup Done")

In [ ]:
import os

# Set your single Kaggle API token
os.environ["KAGGLE_API_TOKEN"] = "KGAT_0b832ef24a29baf72b91c7c6cf473f3b"

import kagglehub

# Download the competition dataset
path = kagglehub.competition_download('ultrasound-nerve-segmentation')
print("Path to competition files:", path)

In [ ]:
# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.competition_download('ultrasound-nerve-segmentation')

print("Path to competition files:", path)

In [ ]:
import os
import kagglehub

# 1. Download dataset via kagglehub
# kagglehub returns the exact local path on your machine/colab
DATA_DIR = kagglehub.competition_download('ultrasound-nerve-segmentation')

print("Data downloaded to:", DATA_DIR)

# 2. Set TRAIN_DIR and TEST_DIR relative to DATA_DIR
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

# 3. Check training files count safely
if os.path.exists(TRAIN_DIR):
    print("Train files count:", len(os.listdir(TRAIN_DIR)))
else:
    # Some competitions flatten images or put them directly in DATA_DIR
    print("All files in dataset directory:", os.listdir(DATA_DIR)[:10])

In [ ]:
images = []
masks = []

for file in os.listdir(TRAIN_DIR):
    if "_mask" not in file:
        img_path = os.path.join(TRAIN_DIR, file)
        mask_path = os.path.join(TRAIN_DIR, file.replace(".tif", "_mask.tif"))

        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            # ✅ KEEP ONLY NON-EMPTY MASKS
            if np.max(mask) > 0:
                images.append(img_path)
                masks.append(mask_path)

print("Filtered pairs (with nerve):", len(images))

In [ ]:
from sklearn.model_selection import train_test_split

train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    images, masks, test_size=0.2, random_state=42
)

print("Train:", len(train_imgs))
print("Val:", len(val_imgs))

In [ ]:
class NerveDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = cv2.imread(self.images[idx], cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(self.masks[idx], cv2.IMREAD_GRAYSCALE)

        img = cv2.resize(img, (128, 128))
        mask = cv2.resize(mask, (128, 128))

        img = (img - np.mean(img)) / (np.std(img) + 1e-6)
        mask = mask / 255.0

        img = np.expand_dims(img, axis=0)
        mask = np.expand_dims(mask, axis=0)

        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

In [ ]:
train_dataset = NerveDataset(train_imgs, train_masks)
val_dataset = NerveDataset(val_imgs, val_masks)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

print("DataLoader Ready")

In [ ]:
img, mask = train_dataset[0]

plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(img.squeeze(), cmap='gray')
plt.title("Image")

plt.subplot(1,2,2)
plt.imshow(mask.squeeze(), cmap='gray')
plt.title("Mask")

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=1,
    classes=1,
    activation=None,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model Loaded")

In [ ]:
dice_loss = smp.losses.DiceLoss(mode="binary")
bce_loss = nn.BCEWithLogitsLoss()

def loss_fn(preds, targets):
    return dice_loss(preds, targets) + bce_loss(preds, targets)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Loss function fixed ✅")

In [ ]:
def train_epoch(loader, model, optimizer):
    model.train()
    total_loss = 0

    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)

        preds = model(imgs)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def val_epoch(loader, model):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)

            preds = model(imgs)
            loss = loss_fn(preds, masks)

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):
    train_loss = train_epoch(train_loader, model, optimizer)
    val_loss = val_epoch(val_loader, model)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
model.eval()

img, mask = val_dataset[0]
img = img.unsqueeze(0).to(device)

with torch.no_grad():
    pred = torch.sigmoid(model(img)).cpu().squeeze().numpy()

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img.cpu().squeeze(), cmap='gray')
plt.title("Input")

plt.subplot(1,3,2)
plt.imshow(mask.squeeze(), cmap='gray')
plt.title("Ground Truth")

plt.subplot(1,3,3)
plt.imshow(pred > 0.5, cmap='gray')
plt.title("Prediction")

In [ ]:
count_empty = 0

for m in train_masks:
    mask = cv2.imread(m, cv2.IMREAD_GRAYSCALE)
    if np.max(mask) == 0:
        count_empty += 1

print("Empty masks:", count_empty)
print("Total masks:", len(train_masks))

In [ ]:
def dice_score(pred, target, smooth=1e-6):
    pred = (pred > 0.5).float()
    target = target.float()

    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

In [ ]:
model.eval()
dice_scores = []

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = torch.sigmoid(model(imgs))

        for i in range(len(preds)):
            d = dice_score(preds[i], masks[i])
            dice_scores.append(d.item())

print("Average Dice Score:", np.mean(dice_scores))

DATA AUGUMENTATION

In [ ]:
!pip install -q albumentations

import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
DATA_DIR = "/root/.cache/kagglehub/competitions/ultrasound-nerve-segmentation"
TRAIN_DIR = os.path.join(DATA_DIR, "train")

In [ ]:
images, masks = [], []

for file in os.listdir(TRAIN_DIR):
    if "_mask" not in file:
        img_path = os.path.join(TRAIN_DIR, file)
        mask_path = os.path.join(TRAIN_DIR, file.replace(".tif", "_mask.tif"))

        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if np.max(mask) > 0 or np.random.rand() < 0.3:
                images.append(img_path)
                masks.append(mask_path)

print("Total usable samples:", len(images))

In [ ]:
train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    images, masks, test_size=0.2, random_state=42
)

In [ ]:
train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=20, p=0.5),
    A.ElasticTransform(p=0.3),
    A.GridDistortion(p=0.3),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256, 256),
    ToTensorV2()
])

In [ ]:
class NerveDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = cv2.imread(self.images[idx], cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(self.masks[idx], cv2.IMREAD_GRAYSCALE)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            img = torch.tensor(img).unsqueeze(0)
            mask = torch.tensor(mask).unsqueeze(0)

        img = img.float() / 255.0
        img = (img - img.mean()) / (img.std() + 1e-6)

        mask = (mask > 0).float().unsqueeze(0)

        return img, mask

In [ ]:
train_dataset = NerveDataset(train_imgs, train_masks, train_transform)
val_dataset = NerveDataset(val_imgs, val_masks, val_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.Sigmoid())
        self.relu = nn.ReLU()

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

In [ ]:
class AttUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = self.block(1, 64)
        self.conv2 = self.block(64, 128)
        self.conv3 = self.block(128, 256)
        self.conv4 = self.block(256, 512)

        self.pool = nn.MaxPool2d(2)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.att4 = AttentionBlock(256, 256, 128)
        self.up_conv4 = self.block(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.att3 = AttentionBlock(128, 128, 64)
        self.up_conv3 = self.block(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.att2 = AttentionBlock(64, 64, 32)
        self.up_conv2 = self.block(128, 64)

        self.final = nn.Conv2d(64, 1, 1)

    def block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )

    def forward(self, x):
        c1 = self.conv1(x)
        c2 = self.conv2(self.pool(c1))
        c3 = self.conv3(self.pool(c2))
        c4 = self.conv4(self.pool(c3))

        u4 = self.up4(c4)
        a4 = self.att4(u4, c3)
        u4 = self.up_conv4(torch.cat([u4, a4], dim=1))

        u3 = self.up3(u4)
        a3 = self.att3(u3, c2)
        u3 = self.up_conv3(torch.cat([u3, a3], dim=1))

        u2 = self.up2(u3)
        a2 = self.att2(u2, c1)
        u2 = self.up_conv2(torch.cat([u2, a2], dim=1))

        return self.final(u2)

In [ ]:
model = AttUNet().to(device)

In [ ]:
def dice_loss(pred, target):
    pred = torch.sigmoid(pred)
    smooth = 1e-6
    intersection = (pred * target).sum()
    return 1 - (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def focal_loss(pred, target, alpha=0.8, gamma=2):
    pred = torch.sigmoid(pred)
    bce = nn.functional.binary_cross_entropy(pred, target, reduction='none')
    pt = torch.exp(-bce)
    return (alpha * (1 - pt) ** gamma * bce).mean()

def loss_fn(pred, target):
    return 0.7 * dice_loss(pred, target) + 0.3 * focal_loss(pred, target)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
def dice_score(pred, target):
    pred = (torch.sigmoid(pred) > 0.5).float()
    intersection = (pred * target).sum()
    return (2. * intersection) / (pred.sum() + target.sum() + 1e-6)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

In [ ]:
EPOCHS = 25
best_dice = 0


for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)

        preds = model(imgs)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_dice = []

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)

            val_dice.append(dice_score(preds, masks).item())

    avg_dice = np.mean(val_dice)

    # ✅ Step scheduler ONCE per epoch
    scheduler.step(avg_dice)

    # ✅ Save best model
    if avg_dice > best_dice:
        best_dice = avg_dice
        torch.save(model.state_dict(), "best_model.pth")

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Dice: {avg_dice:.4f}")

In [ ]:
model.eval()

img, mask = val_dataset[0]
img = img.unsqueeze(0).to(device)

with torch.no_grad():
    pred = torch.sigmoid(model(img)).cpu().squeeze().numpy()

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img.cpu().squeeze(), cmap='gray')
plt.title("Input")

plt.subplot(1,3,2)
plt.imshow(mask.squeeze(), cmap='gray')
plt.title("Ground Truth")

plt.subplot(1,3,3)
plt.imshow(pred > 0.5, cmap='gray')
plt.title("Prediction")

In [ ]:
import segmentation_models_pytorch as smp

In [ ]:
model1 = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=1,
    classes=1,
    activation=None
).to(device)

print("Pretrained Model Loaded ✅")
for param in model1.encoder.parameters():
    param.requires_grad = True

In [ ]:
optimizer1 = torch.optim.Adam(model1.parameters(), lr=1e-5, weight_decay=1e-4)

scheduler1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer1, mode='max', factor=0.5, patience=3
)

In [ ]:
bce_loss1 = nn.BCEWithLogitsLoss()

def boundary_loss(pred, target):
    pred = torch.sigmoid(pred)

    laplace = torch.tensor([[0,1,0],[1,-4,1],[0,1,0]], dtype=torch.float32).to(device)
    laplace = laplace.unsqueeze(0).unsqueeze(0)

    edge_pred = torch.nn.functional.conv2d(pred, laplace, padding=1)
    edge_target = torch.nn.functional.conv2d(target, laplace, padding=1)

    return nn.functional.l1_loss(edge_pred, edge_target)

def loss_fn1(pred, target):
    return (
        0.5 * dice_loss(pred, target) +
        0.3 * bce_loss1(pred, target) +
        0.2 * boundary_loss(pred, target)
    )

In [ ]:
A.ElasticTransform(p=0.3),
A.GridDistortion(p=0.3),

In [ ]:
EPOCHS1 = 35
best_dice1 = 0

for epoch in range(EPOCHS1):
    model1.train()
    train_loss1 = 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)

        preds = model1(imgs)
        loss = loss_fn(preds, masks)

        optimizer1.zero_grad()
        loss.backward()
        optimizer1.step()

        train_loss1 += loss.item()

    # Validation
    model1.eval()
    val_dice1 = []

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model1(imgs)

            val_dice1.append(dice_score(preds, masks).item())

    avg_dice1 = np.mean(val_dice1)

    scheduler1.step(avg_dice1)

    if avg_dice1 > best_dice1:
        best_dice1 = avg_dice1
        torch.save(model1.state_dict(), "best_model1.pth")

    print(f"[Model1] Epoch {epoch+1} | Loss: {train_loss1:.4f} | Dice: {avg_dice1:.4f}")

In [ ]:
thresholds = np.arange(0.3, 0.7, 0.05)

best_thresh1 = 0.5
best_dice_thresh1 = 0

model1.eval()

for t in thresholds:
    dices = []

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = torch.sigmoid(model1(imgs))

            preds_bin = (preds > t).float()

            intersection = (preds_bin * masks).sum()
            dice = (2. * intersection) / (preds_bin.sum() + masks.sum() + 1e-6)

            dices.append(dice.item())

    avg = np.mean(dices)

    if avg > best_dice_thresh1:
        best_dice_thresh1 = avg
        best_thresh1 = t

print("Best Threshold:", best_thresh1)
print("Best Dice:", best_dice_thresh1)

In [ ]:
def tta_predict1(model, img):
    preds = []

    for flip in [False, True]:
        if flip:
            img_flip = torch.flip(img, dims=[3])
        else:
            img_flip = img

        pred = torch.sigmoid(model(img_flip))

        if flip:
            pred = torch.flip(pred, dims=[3])

        preds.append(pred)

    return torch.mean(torch.stack(preds), dim=0)

In [ ]:
model1.eval()

img, mask = val_dataset[0]
img = img.unsqueeze(0).to(device)

with torch.no_grad():
    pred1 = tta_predict1(model1, img).cpu().squeeze().numpy()

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img.cpu().squeeze(), cmap='gray')
plt.title("Input")

plt.subplot(1,3,2)
plt.imshow(mask.squeeze(), cmap='gray')
plt.title("Ground Truth")

plt.subplot(1,3,3)
plt.imshow(pred1 > best_thresh1, cmap='gray')
plt.title("Prediction (Model1)")

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, Model
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')
TRAIN_PATH = "/kaggle/input/competitions/ultrasound-nerve-segmentation/train"


import albumentations as A

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.ElasticTransform(p=0.4),
    A.GridDistortion(p=0.3),
    A.RandomBrightnessContrast(p=0.3),
])
def center_crop(img, size=96):
    h, w = img.shape
    startx = w//2 - size//2
    starty = h//2 - size//2
    return img[starty:starty+size, startx:startx+size]
def apply_clahe(img):
    img = (img * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img = clahe.apply(img)
    return img / 255.0
def load_data():
    images, masks = [], []

    files = os.listdir(TRAIN_PATH)
    image_files = [f for f in files if "_mask" not in f]

    for img_file in image_files:
        mask_file = img_file.replace(".tif", "_mask.tif")

        img_path = os.path.join(TRAIN_PATH, img_file)
        mask_path = os.path.join(TRAIN_PATH, mask_file)

        if os.path.exists(mask_path):
            img = cv2.imread(img_path, 0)
            mask = cv2.imread(mask_path, 0)
            aug = transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']

            img = center_crop(img)
            img = cv2.resize(img, (128,128))
            mask = cv2.resize(mask, (128, 128))
            mask = (mask > 127).astype(np.float32)  # 🔥 FIX
            img = apply_clahe(img)

            images.append(img)
            masks.append(mask)

    X = np.array(images).reshape(-1, 128, 128, 1)
    y = np.array(masks).reshape(-1, 128, 128, 1)

    return X, y

X, y = load_data()
print(X.shape, y.shape)

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    return x

def attention_block(x, g, filters):
    theta_x = layers.Conv2D(filters, 1)(x)
    phi_g = layers.Conv2D(filters, 1)(g)

    add = layers.Add()([theta_x, phi_g])
    act = layers.Activation('relu')(add)
    psi = layers.Conv2D(1, 1, activation='sigmoid')(act)

    return layers.Multiply()([x, psi])

def transformer_block(x):
    b, h, w, c = x.shape

    x_flat = layers.Reshape((h*w, c))(x)

    attn = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=32
    )(x_flat, x_flat)

    x = layers.Add()([x_flat, attn])
    x = layers.Reshape((h, w, c))(x)

    return x

def build_model():
    inputs = layers.Input((128, 128, 1))

    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 256)

    t = transformer_block(c4)

    u1 = layers.UpSampling2D()(t)
    a1 = attention_block(c3, u1, 128)
    u1 = layers.Concatenate()([u1, a1])
    c5 = conv_block(u1, 128)

    u2 = layers.UpSampling2D()(c5)
    a2 = attention_block(c2, u2, 64)
    u2 = layers.Concatenate()([u2, a2])
    c6 = conv_block(u2, 64)

    u3 = layers.UpSampling2D()(c6)
    a3 = attention_block(c1, u3, 32)
    u3 = layers.Concatenate()([u3, a3])
    c7 = conv_block(u3, 32)

    outputs = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32')(c7)

    return Model(inputs, outputs)

model = build_model()
model.summary()

In [ ]:
def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    bce_exp = tf.exp(-bce)

    focal = alpha * (1 - bce_exp) ** gamma * bce
    return tf.reduce_mean(focal)
def tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    TP = tf.reduce_sum(y_true_f * y_pred_f)
    FP = tf.reduce_sum((1 - y_true_f) * y_pred_f)
    FN = tf.reduce_sum(y_true_f * (1 - y_pred_f))

    return 1 - (TP + 1) / (TP + alpha*FP + beta*FN + 1)
def combined_loss(y_true, y_pred):
    return focal_loss(y_true, y_pred) + dice_loss(y_true, y_pred)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),  # 🔥 increase LR slightly
    loss=tversky_loss,
    metrics=[dice_coef]
)

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6
    ),
    tf.keras.callbacks.EarlyStopping(
        patience=8,
        restore_best_weights=True
    )
]

history = model.fit(
    X, y,
    batch_size=2,
    epochs=30,
    validation_split=0.1,
    callbacks=callbacks
)

In [ ]:
TEST_PATH = "/kaggle/input/competitions/ultrasound-nerve-segmentation/test"

def load_test():
    images, names = [], []

    for file in os.listdir(TEST_PATH):
        img = cv2.imread(os.path.join(TEST_PATH, file), 0)
        img = cv2.resize(img, (128, 128)) / 255.0

        images.append(img)
        names.append(file)

    X_test = np.array(images).reshape(-1, 128, 128, 1)
    return X_test, names

X_test, names = load_test()
preds = model.predict(X_test)

In [ ]:
import cv2

def clean_mask(mask):
    mask = mask.astype(np.uint8)
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return mask

In [ ]:
def visualize_predictions(X_test, preds, n=5):
    for i in range(n):
        plt.figure(figsize=(12,4))

        plt.subplot(1,3,1)
        plt.imshow(X_test[i].squeeze(), cmap='gray')
        plt.title("Original")
        plt.axis('off')

        plt.subplot(1,3,2)
        plt.imshow(preds[i].squeeze(), cmap='gray')
        plt.title("Raw Prediction")
        plt.axis('off')

        plt.subplot(1,3,3)
        mask = clean_mask((preds[i].squeeze() > 0.3))
        plt.imshow(mask, cmap='gray')
        plt.title("Binary Mask")
        plt.axis('off')

        plt.show()

visualize_predictions(X_test, preds)
def overlay(img, pred):
    plt.imshow(img.squeeze(), cmap='gray')
    plt.imshow((pred.squeeze()>0.5), alpha=0.4, cmap='jet')
    plt.axis('off')
    plt.show()

overlay(X_test[0], preds[0])

In [ ]:
# =========================
# IMPORTS
# =========================
import os, cv2, numpy as np, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')

# =========================
# PATHS
# =========================
TRAIN_PATH = "/kaggle/input/competitions/ultrasound-nerve-segmentation/train"
TEST_PATH  = "/kaggle/input/competitions/ultrasound-nerve-segmentation/test"

# =========================
# CLAHE (SAFE VERSION)
# =========================
def apply_clahe(img):
    img = (img * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img = clahe.apply(img)
    return img / 255.0

# =========================
# DATA LOADER (FIXED)
# =========================
def load_data():
    images, masks = [], []
    files = os.listdir(TRAIN_PATH)
    image_files = [f for f in files if "_mask" not in f]

    for img_file in image_files:
        mask_file = img_file.replace(".tif", "_mask.tif")

        img_path = os.path.join(TRAIN_PATH, img_file)
        mask_path = os.path.join(TRAIN_PATH, mask_file)

        if os.path.exists(mask_path):
            img = cv2.imread(img_path, 0)
            mask = cv2.imread(mask_path, 0)

            img = cv2.resize(img, (128,128)) / 255.0
            img = apply_clahe(img)  # 🔥 FIXED CLAHE

            mask = cv2.resize(mask, (128,128))
            mask = (mask > 127).astype(np.float32)

            images.append(img)
            masks.append(mask)

    X = np.array(images).reshape(-1,128,128,1)
    y = np.array(masks).reshape(-1,128,128,1)
    return X, y

X, y = load_data()
print("Data:", X.shape, y.shape)

# =========================
# MODEL (STABLE VERSION)
# =========================
def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(f, 3, padding='same', activation='relu')(x)
    return x

def build_model():
    inputs = layers.Input((128,128,1))

    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 256)

    u1 = layers.UpSampling2D()(c4)
    u1 = layers.Concatenate()([u1, c3])
    c5 = conv_block(u1, 128)

    u2 = layers.UpSampling2D()(c5)
    u2 = layers.Concatenate()([u2, c2])
    c6 = conv_block(u2, 64)

    u3 = layers.UpSampling2D()(c6)
    u3 = layers.Concatenate()([u3, c1])
    c7 = conv_block(u3, 32)

    outputs = layers.Conv2D(1,1,activation='sigmoid', dtype='float32')(c7)

    return Model(inputs, outputs)

model = build_model()
model.summary()

# =========================
# LOSS + METRIC (STABLE)
# =========================
def dice_coef(y_true, y_pred):
    smooth = 1.
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.*intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

# =========================
# COMPILE
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=bce_dice_loss,
    metrics=[dice_coef]
)

# =========================
# CALLBACKS
# =========================
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
    tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)
]

# =========================
# TRAIN
# =========================
history = model.fit(
    X, y,
    batch_size=4,
    epochs=30,
    validation_split=0.1,
    callbacks=callbacks
)

# =========================
# TEST LOADER
# =========================
def load_test():
    images, names = [], []
    for file in os.listdir(TEST_PATH):
        img = cv2.imread(os.path.join(TEST_PATH, file), 0)
        img = cv2.resize(img, (128,128)) / 255.0
        img = apply_clahe(img)
        images.append(img)
        names.append(file)

    X_test = np.array(images).reshape(-1,128,128,1)
    return X_test, names

X_test, names = load_test()
preds = model.predict(X_test)

# =========================
# VISUALIZATION
# =========================
def visualize(X, preds, n=5):
    for i in range(n):
        plt.figure(figsize=(12,4))

        plt.subplot(1,3,1)
        plt.imshow(X[i].squeeze(), cmap='gray')
        plt.title("Original")
        plt.axis('off')

        plt.subplot(1,3,2)
        plt.imshow(preds[i].squeeze(), cmap='gray')
        plt.title("Raw")
        plt.axis('off')

        plt.subplot(1,3,3)
        mask = (preds[i].squeeze() > 0.3)
        plt.imshow(mask, cmap='gray')
        plt.title("Mask")
        plt.axis('off')

        plt.show()

visualize(X_test, preds)

# =========================
# SUBMISSION
# =========================
def rle_encode(mask):
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

pred_masks = (preds > 0.3).astype(np.uint8)

rles = [rle_encode(pred_masks[i].squeeze()) for i in range(len(pred_masks))]

df = pd.DataFrame({'img': names, 'pixels': rles})
df.to_csv("submission.csv", index=False)

print("✅ Done: Training + Visualization + Submission Ready")

In [ ]:
# =========================
# IMPORTS
# =========================
import os, cv2, numpy as np, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')

TRAIN_PATH = "/kaggle/input/competitions/ultrasound-nerve-segmentation/train"
TEST_PATH  = "/kaggle/input/competitions/ultrasound-nerve-segmentation/test"

# =========================
# CLAHE
# =========================
def apply_clahe(img):
    img = (img * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(2.0, (8,8))
    return clahe.apply(img) / 255.0

# =========================
# LOAD DATA + AUGMENT
# =========================
def load_data():
    images, masks = [], []
    files = os.listdir(TRAIN_PATH)
    image_files = [f for f in files if "_mask" not in f]

    for img_file in image_files:
        mask_file = img_file.replace(".tif", "_mask.tif")

        img = cv2.imread(os.path.join(TRAIN_PATH, img_file), 0)
        mask = cv2.imread(os.path.join(TRAIN_PATH, mask_file), 0)

        img = cv2.resize(img, (128,128))/255.0
        img = apply_clahe(img)

        mask = cv2.resize(mask, (128,128))
        mask = (mask > 127).astype(np.float32)

        # ORIGINAL
        images.append(img)
        masks.append(mask)

        # AUGMENT (flip)
        images.append(np.fliplr(img))
        masks.append(np.fliplr(mask))

    X = np.array(images).reshape(-1,128,128,1)
    y = np.array(masks).reshape(-1,128,128,1)

    # SHUFFLE (IMPORTANT)
    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

X, y = load_data()

# =========================
# MODEL (Attention U-Net)
# =========================
def conv_block(x, f):
    x = layers.Conv2D(f,3,padding='same',activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(f,3,padding='same',activation='relu')(x)
    return x

def attention(x):
    g = layers.Conv2D(1,1,activation='sigmoid')(x)
    return layers.Multiply()([x,g])

def build_model():
    inp = layers.Input((128,128,1))

    c1 = conv_block(inp,32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1,64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2,128)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3,256)
    c4 = attention(c4)

    u1 = layers.UpSampling2D()(c4)
    u1 = layers.Concatenate()([u1,c3])
    c5 = conv_block(u1,128)

    u2 = layers.UpSampling2D()(c5)
    u2 = layers.Concatenate()([u2,c2])
    c6 = conv_block(u2,64)

    u3 = layers.UpSampling2D()(c6)
    u3 = layers.Concatenate()([u3,c1])
    c7 = conv_block(u3,32)

    out = layers.Conv2D(1,1,activation='sigmoid',dtype='float32')(c7)
    return Model(inp,out)

model = build_model()

# =========================
# LOSS
# =========================
def dice_coef(y_true,y_pred):
    y_true = tf.reshape(y_true,[-1])
    y_pred = tf.reshape(y_pred,[-1])
    inter = tf.reduce_sum(y_true*y_pred)
    return (2*inter+1)/(tf.reduce_sum(y_true)+tf.reduce_sum(y_pred)+1)

def dice_loss(y_true,y_pred):
    return 1 - dice_coef(y_true,y_pred)

def focal_loss(y_true,y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true,y_pred)
    pt = tf.exp(-bce)
    return tf.reduce_mean(0.25*(1-pt)**2 * bce)

def final_loss(y_true,y_pred):
    return focal_loss(y_true,y_pred) + dice_loss(y_true,y_pred)

# =========================
# COMPILE
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=final_loss,
    metrics=[dice_coef]
)

# =========================
# TRAIN
# =========================
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(patience=3,factor=0.5),
    tf.keras.callbacks.EarlyStopping(patience=6,restore_best_weights=True)
]

model.fit(
    X,y,
    batch_size=8,
    epochs=30,
    validation_split=0.1,
    callbacks=callbacks
)

# =========================
# LOAD TEST
# =========================
def load_test():
    imgs, names = [], []
    for f in os.listdir(TEST_PATH):
        img = cv2.imread(os.path.join(TEST_PATH,f),0)
        img = cv2.resize(img,(128,128))/255.0
        img = apply_clahe(img)
        imgs.append(img)
        names.append(f)
    return np.array(imgs).reshape(-1,128,128,1), names

X_test, names = load_test()

# =========================
# FAST TTA (VECTOR)
# =========================
pred1 = model.predict(X_test, batch_size=32)

flip = np.flip(X_test, axis=2)
pred2 = model.predict(flip, batch_size=32)
pred2 = np.flip(pred2, axis=2)

preds = (pred1 + pred2) / 2

# =========================
# CLEAN MASK
# =========================
def clean(mask):
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    return cv2.dilate(mask, kernel, iterations=1)

# =========================
# VISUALIZE
# =========================
for i in range(5):
    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(X_test[i].squeeze(),cmap='gray')
    plt.title("Original"); plt.axis('off')

    plt.subplot(1,3,2)
    plt.imshow(preds[i].squeeze(),cmap='gray')
    plt.title("Raw"); plt.axis('off')

    plt.subplot(1,3,3)
    plt.imshow(clean(preds[i].squeeze()>0.25),cmap='gray')
    plt.title("Mask"); plt.axis('off')

    plt.show()

# =========================
# SUBMISSION
# =========================
def rle(mask):
    pixels = mask.flatten()
    pixels = np.concatenate([[0],pixels,[0]])
    runs = np.where(pixels[1:]!=pixels[:-1])[0]+1
    runs[1::2] -= runs[::2]
    return ' '.join(map(str,runs))

pred_masks = [clean(p.squeeze()>0.25) for p in preds]
rles = [rle(m) for m in pred_masks]

df = pd.DataFrame({'img':names,'pixels':rles})
df.to_csv("submission.csv",index=False)

print("✅ FINAL PIPELINE READY")

In [ ]:
# =========================
# IMPORTS
# =========================
import os, cv2, numpy as np, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')

TRAIN_PATH = "/kaggle/input/competitions/ultrasound-nerve-segmentation/train"
TEST_PATH  = "/kaggle/input/competitions/ultrasound-nerve-segmentation/test"

# =========================
# CLAHE (Enhance Ultrasound)
# =========================
def apply_clahe(img):
    img = (img * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(2.0, (8,8))
    return clahe.apply(img) / 255.0

# =========================
# LOAD + AUGMENT DATA
# =========================
def load_data():
    images, masks = [], []
    files = os.listdir(TRAIN_PATH)
    image_files = [f for f in files if "_mask" not in f]

    for img_file in image_files:
        mask_file = img_file.replace(".tif", "_mask.tif")

        img = cv2.imread(os.path.join(TRAIN_PATH, img_file), 0)
        mask = cv2.imread(os.path.join(TRAIN_PATH, mask_file), 0)

        img = cv2.resize(img, (128,128)) / 255.0
        img = apply_clahe(img)

        mask = cv2.resize(mask, (128,128))
        mask = (mask > 127).astype(np.float32)

        # original
        images.append(img)
        masks.append(mask)

        # augment: horizontal flip
        images.append(np.fliplr(img))
        masks.append(np.fliplr(mask))

    X = np.array(images).reshape(-1,128,128,1)
    y = np.array(masks).reshape(-1,128,128,1)

    # shuffle (important)
    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

X, y = load_data()

# =========================
# MODEL (Attention U-Net)
# =========================
def conv_block(x, f):
    x = layers.Conv2D(f,3,padding='same',activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(f,3,padding='same',activation='relu')(x)
    return x

def attention(x):
    g = layers.Conv2D(1,1,activation='sigmoid')(x)
    return layers.Multiply()([x, g])

def build_model():
    inp = layers.Input((128,128,1))

    c1 = conv_block(inp,32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1,64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2,128)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3,256)
    c4 = attention(c4)

    u1 = layers.UpSampling2D()(c4)
    u1 = layers.Concatenate()([u1,c3])
    c5 = conv_block(u1,128)

    u2 = layers.UpSampling2D()(c5)
    u2 = layers.Concatenate()([u2,c2])
    c6 = conv_block(u2,64)

    u3 = layers.UpSampling2D()(c6)
    u3 = layers.Concatenate()([u3,c1])
    c7 = conv_block(u3,32)

    out = layers.Conv2D(1,1,activation='sigmoid',dtype='float32')(c7)

    return Model(inp,out)

model = build_model()
model.summary()

# =========================
# LOSS (FOCAL + DICE)
# =========================
def dice_coef(y_true,y_pred):
    y_true = tf.reshape(y_true,[-1])
    y_pred = tf.reshape(y_pred,[-1])
    inter = tf.reduce_sum(y_true*y_pred)
    return (2*inter+1)/(tf.reduce_sum(y_true)+tf.reduce_sum(y_pred)+1)

def dice_loss(y_true,y_pred):
    return 1 - dice_coef(y_true,y_pred)

def focal_loss(y_true,y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true,y_pred)
    pt = tf.exp(-bce)
    return tf.reduce_mean(0.25*(1-pt)**2 * bce)

def final_loss(y_true,y_pred):
    return focal_loss(y_true,y_pred) + dice_loss(y_true,y_pred)

# =========================
# COMPILE
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=final_loss,
    metrics=[dice_coef]
)

# =========================
# TRAIN
# =========================
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(patience=3,factor=0.5),
    tf.keras.callbacks.EarlyStopping(patience=6,restore_best_weights=True)
]

model.fit(
    X, y,
    batch_size=8,
    epochs=30,
    validation_split=0.1,
    callbacks=callbacks
)

# =========================
# LOAD TEST DATA
# =========================
def load_test():
    imgs, names = [], []
    for f in os.listdir(TEST_PATH):
        img = cv2.imread(os.path.join(TEST_PATH,f),0)
        img = cv2.resize(img,(128,128))/255.0
        img = apply_clahe(img)
        imgs.append(img)
        names.append(f)
    return np.array(imgs).reshape(-1,128,128,1), names

X_test, names = load_test()

# =========================
# FAST TTA (VECTOR)
# =========================
pred1 = model.predict(X_test, batch_size=32)

flip = np.flip(X_test, axis=2)
pred2 = model.predict(flip, batch_size=32)
pred2 = np.flip(pred2, axis=2)

preds = (pred1 + pred2) / 2

# =========================
# CLEAN MASK
# =========================
def clean(mask):
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    return cv2.dilate(mask, kernel, iterations=1)

# =========================
# VISUALIZATION
# =========================
for i in range(5):
    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(X_test[i].squeeze(),cmap='gray')
    plt.title("Original"); plt.axis('off')

    plt.subplot(1,3,2)
    plt.imshow(preds[i].squeeze(),cmap='gray')
    plt.title("Raw"); plt.axis('off')

    plt.subplot(1,3,3)
    plt.imshow(clean(preds[i].squeeze()>0.25),cmap='gray')
    plt.title("Mask"); plt.axis('off')

    plt.show()

# =========================
# SUBMISSION (FIXED)
# =========================
def rle(mask):
    pixels = mask.flatten()
    pixels = np.concatenate([[0],pixels,[0]])
    runs = np.where(pixels[1:]!=pixels[:-1])[0]+1
    runs[1::2] -= runs[::2]
    return ' '.join(map(str,runs))

pred_masks = [clean(p.squeeze()>0.25) for p in preds]
rles = [rle(m) for m in pred_masks]

df = pd.DataFrame({'img':names,'pixels':rles})
df.to_csv("submission.csv",index=False)

print("✅ FINAL PIPELINE READY 🚀")

In [ ]:
# =========================
# IMPORTS
# =========================
import os, cv2, numpy as np, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from tensorflow.keras import mixed_precision
from sklearn.metrics import classification_report

mixed_precision.set_global_policy('mixed_float16')

TRAIN_PATH = "/kaggle/input/competitions/ultrasound-nerve-segmentation/train"
TEST_PATH  = "/kaggle/input/competitions/ultrasound-nerve-segmentation/test"

# =========================
# CLAHE
# =========================
def apply_clahe(img):
    img = (img * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(2.0, (8,8))
    return clahe.apply(img) / 255.0

# =========================
# LOAD DATA + AUGMENT
# =========================
def load_data():
    images, masks = [], []
    files = os.listdir(TRAIN_PATH)
    image_files = [f for f in files if "_mask" not in f]

    for img_file in image_files:
        mask_file = img_file.replace(".tif", "_mask.tif")

        img = cv2.imread(os.path.join(TRAIN_PATH, img_file), 0)
        mask = cv2.imread(os.path.join(TRAIN_PATH, mask_file), 0)

        img = cv2.resize(img, (128,128))/255.0
        img = apply_clahe(img)

        mask = cv2.resize(mask, (128,128))
        mask = (mask > 127).astype(np.float32)

        images.append(img)
        masks.append(mask)

        # augmentation
        images.append(np.fliplr(img))
        masks.append(np.fliplr(mask))

    X = np.array(images).reshape(-1,128,128,1)
    y = np.array(masks).reshape(-1,128,128,1)

    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

X, y = load_data()

# =========================
# MODEL
# =========================
def conv_block(x, f):
    x = layers.Conv2D(f,3,padding='same',activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(f,3,padding='same',activation='relu')(x)
    return x

def attention(x):
    g = layers.Conv2D(1,1,activation='sigmoid')(x)
    return layers.Multiply()([x, g])

def build_model():
    inp = layers.Input((128,128,1))

    c1 = conv_block(inp,32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1,64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2,128)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3,256)
    c4 = attention(c4)

    u1 = layers.UpSampling2D()(c4)
    u1 = layers.Concatenate()([u1,c3])
    c5 = conv_block(u1,128)

    u2 = layers.UpSampling2D()(c5)
    u2 = layers.Concatenate()([u2,c2])
    c6 = conv_block(u2,64)

    u3 = layers.UpSampling2D()(c6)
    u3 = layers.Concatenate()([u3,c1])
    c7 = conv_block(u3,32)

    out = layers.Conv2D(1,1,activation='sigmoid',dtype='float32')(c7)

    return Model(inp,out)

model = build_model()

# =========================
# LOSS
# =========================
def dice_coef(y_true,y_pred):
    y_true = tf.reshape(y_true,[-1])
    y_pred = tf.reshape(y_pred,[-1])
    inter = tf.reduce_sum(y_true*y_pred)
    return (2*inter+1)/(tf.reduce_sum(y_true)+tf.reduce_sum(y_pred)+1)

def dice_loss(y_true,y_pred):
    return 1 - dice_coef(y_true,y_pred)

def focal_loss(y_true,y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true,y_pred)
    pt = tf.exp(-bce)
    return tf.reduce_mean(0.25*(1-pt)**2 * bce)

def final_loss(y_true,y_pred):
    return focal_loss(y_true,y_pred) + dice_loss(y_true,y_pred)

# =========================
# COMPILE
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=final_loss,
    metrics=[dice_coef]
)

# =========================
# TRAIN
# =========================
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(patience=3,factor=0.5),
    tf.keras.callbacks.EarlyStopping(patience=6,restore_best_weights=True)
]

model.fit(
    X,y,
    batch_size=8,
    epochs=30,
    validation_split=0.1,
    callbacks=callbacks
)

# =========================
# CLASSIFICATION REPORT (PIXEL LEVEL)
# =========================
val_split = int(0.9 * len(X))
X_val, y_val = X[val_split:], y[val_split:]

pred_val = model.predict(X_val, batch_size=32)

y_true_flat = y_val.flatten()
y_pred_flat = (pred_val.flatten() > 0.25).astype(int)

print("\n📊 CLASSIFICATION REPORT (PIXEL LEVEL):\n")
print(classification_report(y_true_flat, y_pred_flat))

# =========================
# LOAD TEST
# =========================
def load_test():
    imgs, names = [], []
    for f in os.listdir(TEST_PATH):
        img = cv2.imread(os.path.join(TEST_PATH,f),0)
        img = cv2.resize(img,(128,128))/255.0
        img = apply_clahe(img)
        imgs.append(img)
        names.append(f)
    return np.array(imgs).reshape(-1,128,128,1), names

X_test, names = load_test()

# =========================
# FAST TTA
# =========================
pred1 = model.predict(X_test, batch_size=32)

flip = np.flip(X_test, axis=2)
pred2 = model.predict(flip, batch_size=32)
pred2 = np.flip(pred2, axis=2)

preds = (pred1 + pred2) / 2

# =========================
# CLEAN MASK
# =========================
def clean(mask):
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    return cv2.dilate(mask, kernel, iterations=1)

# =========================
# VISUALIZE
# =========================
for i in range(5):
    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(X_test[i].squeeze(),cmap='gray')
    plt.title("Original"); plt.axis('off')

    plt.subplot(1,3,2)
    plt.imshow(preds[i].squeeze(),cmap='gray')
    plt.title("Raw"); plt.axis('off')

    plt.subplot(1,3,3)
    plt.imshow(clean(preds[i].squeeze()>0.25),cmap='gray')
    plt.title("Mask"); plt.axis('off')

    plt.show()

# =========================
# SUBMISSION
# =========================
def rle(mask):
    pixels = mask.flatten()
    pixels = np.concatenate([[0],pixels,[0]])
    runs = np.where(pixels[1:]!=pixels[:-1])[0]+1
    runs[1::2] -= runs[::2]
    return ' '.join(map(str,runs))

pred_masks = [clean(p.squeeze()>0.25) for p in preds]
rles = [rle(m) for m in pred_masks]

df = pd.DataFrame({'img':names,'pixels':rles})
df.to_csv("submission.csv",index=False)

print("\n✅ FINAL PIPELINE + REPORT READY 🚀")

SOTA

In [ ]:
# =========================
# IMPORTS
# =========================
import os, cv2, numpy as np, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import mixed_precision
from sklearn.metrics import classification_report

mixed_precision.set_global_policy('mixed_float16')

TRAIN_PATH = "/kaggle/input/competitions/ultrasound-nerve-segmentation/train"
TEST_PATH  = "/kaggle/input/competitions/ultrasound-nerve-segmentation/test"

# =========================
# CLAHE
# =========================
def apply_clahe(img):
    img = (img * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(2.0, (8,8))
    return clahe.apply(img) / 255.0

# =========================
# LOAD DATA
# =========================
def load_data():
    images, masks = [], []
    files = os.listdir(TRAIN_PATH)
    image_files = [f for f in files if "_mask" not in f]

    for img_file in image_files:
        mask_file = img_file.replace(".tif", "_mask.tif")

        img = cv2.imread(os.path.join(TRAIN_PATH, img_file), 0)
        mask = cv2.imread(os.path.join(TRAIN_PATH, mask_file), 0)

        img = cv2.resize(img, (128,128))/255.0
        img = apply_clahe(img)

        mask = cv2.resize(mask, (128,128))
        mask = (mask > 127).astype(np.float32)

        # ORIGINAL
        images.append(img)
        masks.append(mask)

        # AUGMENT
        images.append(np.fliplr(img))
        masks.append(np.fliplr(mask))

    X = np.array(images)
    y = np.array(masks)

    X = X[...,np.newaxis] # for EfficientNet
    y = y[...,np.newaxis]

    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

X, y = load_data()

# =========================
# ATTENTION BLOCK
# =========================
def attention_block(x, g):
    g1 = layers.Conv2D(64,1)(g)
    x1 = layers.Conv2D(64,1)(x)

    psi = layers.Activation('relu')(g1 + x1)
    psi = layers.Conv2D(1,1,activation='sigmoid')(psi)

    return layers.Multiply()([x, psi])

# =========================
# DECODER BLOCK
# =========================
def decoder_block(x, skip, filters):
    x = layers.UpSampling2D()(x)
    skip = attention_block(skip, x)
    x = layers.Concatenate()([x, skip])

    x = layers.Conv2D(filters,3,padding='same',activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters,3,padding='same',activation='relu')(x)
    return x

# =========================
# BUILD SOTA MODEL
# =========================
# =========================
# BUILD MODEL (NO PRETRAIN)
# =========================
def build_model():
    inputs = layers.Input((128,128,1))

    # Encoder
    c1 = layers.Conv2D(32,3,padding='same',activation='relu')(inputs)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Conv2D(32,3,padding='same',activation='relu')(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(64,3,padding='same',activation='relu')(p1)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Conv2D(64,3,padding='same',activation='relu')(c2)
    p2 = layers.MaxPooling2D()(c2)

    c3 = layers.Conv2D(128,3,padding='same',activation='relu')(p2)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Conv2D(128,3,padding='same',activation='relu')(c3)
    p3 = layers.MaxPooling2D()(c3)

    # Bottleneck
    b = layers.Conv2D(256,3,padding='same',activation='relu')(p3)
    b = layers.Conv2D(256,3,padding='same',activation='relu')(b)

    # Decoder
    u1 = layers.UpSampling2D()(b)
    u1 = layers.Concatenate()([u1,c3])
    c4 = layers.Conv2D(128,3,padding='same',activation='relu')(u1)
    c4 = layers.Conv2D(128,3,padding='same',activation='relu')(c4)

    u2 = layers.UpSampling2D()(c4)
    u2 = layers.Concatenate()([u2,c2])
    c5 = layers.Conv2D(64,3,padding='same',activation='relu')(u2)
    c5 = layers.Conv2D(64,3,padding='same',activation='relu')(c5)

    u3 = layers.UpSampling2D()(c5)
    u3 = layers.Concatenate()([u3,c1])
    c6 = layers.Conv2D(32,3,padding='same',activation='relu')(u3)
    c6 = layers.Conv2D(32,3,padding='same',activation='relu')(c6)

    outputs = layers.Conv2D(1,1,activation='sigmoid')(c6)

    return Model(inputs, outputs)

model = build_model()
model.summary()

# =========================
# LOSS
# =========================
def dice_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    smooth = 1.
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / \
           (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def focal_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)   # 🔥 FIX
    y_pred = tf.cast(y_pred, tf.float32)   # 🔥 FIX

    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    pt = tf.exp(-bce)
    return tf.reduce_mean(0.75 * (1 - pt)**2 * bce)

def weighted_bce(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # avoid log(0)
    epsilon = 1e-7
    y_pred = tf.clip_by_value(y_pred, epsilon, 1 - epsilon)

    # manual BCE (no shape issues)
    bce = -(y_true * tf.math.log(y_pred) +
            (1 - y_true) * tf.math.log(1 - y_pred))

    # class weights
    weights = y_true * 10.0 + 1.0

    return tf.reduce_mean(bce * weights)

def final_loss(y_true, y_pred):
    return dice_loss(y_true, y_pred) + weighted_bce(y_true, y_pred)

# =========================
# COMPILE
# =========================
def dice_coef(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)   # 🔥 FIX
    y_pred = tf.cast(y_pred, tf.float32)   # 🔥 FIX

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    inter = tf.reduce_sum(y_true * y_pred)
    return (2 * inter + 1) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + 1)

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),
    loss=final_loss,
    metrics=[dice_coef]
)

# =========================
# TRAIN
# =========================
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5),
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
]

history = model.fit(
    X, y,
    batch_size=4,   # 🔥 stable
    epochs=30,
    validation_split=0.1
)

# =========================
# VALIDATION REPORT
# =========================
val_split = int(0.9 * len(X))
X_val, y_val = X[val_split:], y[val_split:]

pred_val = model.predict(X_val, batch_size=32)

y_true_flat = y_val.flatten()
y_pred_flat = (pred_val.flatten() > 0.3).astype(int)

print("\n📊 CLASSIFICATION REPORT:\n")
print(classification_report(y_true_flat, y_pred_flat))

# =========================
# LOAD TEST
# =========================
def load_test():
    imgs, names = [], []
    for f in os.listdir(TEST_PATH):
        img = cv2.imread(os.path.join(TEST_PATH,f),0)
        img = cv2.resize(img,(128,128))/255.0
        img = apply_clahe(img)

        imgs.append(img)
        names.append(f)

    imgs = np.array(imgs)[..., np.newaxis]   # ✅ FIX HERE
    return imgs, names

X_test, names = load_test()

# =========================
# FAST TTA
# =========================
pred1 = model.predict(X_test, batch_size=32)

flip = np.flip(X_test, axis=2)
pred2 = model.predict(flip, batch_size=32)
pred2 = np.flip(pred2, axis=2)

preds = (pred1 + pred2) / 2

# =========================
# CLEAN MASK
# =========================
def clean(mask):
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    mask = cv2.dilate(mask, kernel, iterations=1)
    return mask

# =========================
# SUBMISSION
# =========================
def rle(mask):
    pixels = mask.flatten()
    pixels = np.concatenate([[0],pixels,[0]])
    runs = np.where(pixels[1:]!=pixels[:-1])[0]+1
    runs[1::2] -= runs[::2]
    return ' '.join(map(str,runs))

pred_masks = [clean(p.squeeze()>0.3) for p in preds]
rles = [rle(m) for m in pred_masks]

df = pd.DataFrame({'img':names,'pixels':rles})
df.to_csv("submission.csv",index=False)

# =========================
# TEST IMAGE VISUALIZATION
# =========================
import matplotlib.pyplot as plt

def show_predictions(X_test, preds, n=6):
    for i in range(n):
        plt.figure(figsize=(12,4))

        # Original
        plt.subplot(1,3,1)
        plt.imshow(X_test[i].squeeze(), cmap='gray')
        plt.title("Original")
        plt.axis('off')

        # Raw prediction
        plt.subplot(1,3,2)
        plt.imshow(preds[i].squeeze(), cmap='gray')
        plt.title("Raw Prediction")
        plt.axis('off')

        # Thresholded mask
        mask = (preds[i].squeeze() > 0.25)

        plt.subplot(1,3,3)
        plt.imshow(mask, cmap='gray')
        plt.title("Final Mask")
        plt.axis('off')

        plt.show()

def overlay_prediction(img, pred):
    mask = (pred.squeeze() > 0.3).astype(np.uint8)

    img = (img.squeeze()*255).astype(np.uint8)
    img_color = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    img_color[:,:,1] = np.maximum(img_color[:,:,1], mask*255)  # green overlay

    return img_color

# Show overlay
for i in range(5):
    overlay = overlay_prediction(X_test[i], preds[i])
    plt.imshow(overlay)
    plt.title("Overlay Prediction")
    plt.axis('off')
    plt.show()
# CALL FUNCTION
show_predictions(X_test, preds, n=8)

print("\n🔥 SOTA MODEL READY 🔥")

> **Attention U-Net**

In [ ]:
# =========================
# IMPORTS
# =========================
import os, cv2, numpy as np, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from tensorflow.keras import mixed_precision
from sklearn.metrics import classification_report

mixed_precision.set_global_policy('mixed_float16')

TRAIN_PATH = "/kaggle/input/competitions/ultrasound-nerve-segmentation/train"
TEST_PATH  = "/kaggle/input/competitions/ultrasound-nerve-segmentation/test"

# =========================
# CLAHE (enhances ultrasound)
# =========================
def apply_clahe(img):
    img = (img * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(2.0, (8,8))
    return clahe.apply(img) / 255.0

# =========================
# LOAD DATA
# =========================
def load_data():
    images, masks = [], []
    files = os.listdir(TRAIN_PATH)

    for f in files:
        if "_mask" in f:
            continue

        mask_file = f.replace(".tif","_mask.tif")

        img = cv2.imread(os.path.join(TRAIN_PATH,f),0)
        mask = cv2.imread(os.path.join(TRAIN_PATH,mask_file),0)

        img = cv2.resize(img,(128,128))/255.0
        img = apply_clahe(img)

        mask = cv2.resize(mask,(128,128))
        mask = (mask > 127).astype(np.float32)

        # original
        images.append(img)
        masks.append(mask)

        # flip augment
        images.append(np.fliplr(img))
        masks.append(np.fliplr(mask))

    X = np.array(images)[...,np.newaxis]
    y = np.array(masks)[...,np.newaxis]

    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

X, y = load_data()
print(X.shape, y.shape)

# =========================
# RESIDUAL BLOCK
# =========================
def res_block(x, filters):
    shortcut = x

    x = layers.Conv2D(filters,3,padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters,3,padding='same')(x)

    shortcut = layers.Conv2D(filters,1,padding='same')(shortcut)

    return layers.Activation('relu')(x + shortcut)

# =========================
# ATTENTION BLOCK
# =========================
def attention_block(x, g):
    g1 = layers.Conv2D(64,1)(g)
    x1 = layers.Conv2D(64,1)(x)

    psi = layers.Activation('relu')(g1 + x1)
    psi = layers.Conv2D(1,1,activation='sigmoid')(psi)

    return layers.Multiply()([x, psi])

# =========================
# DECODER BLOCK
# =========================
def decoder_block(x, skip, filters):
    x = layers.UpSampling2D()(x)
    skip = attention_block(skip, x)
    x = layers.Concatenate()([x, skip])
    x = res_block(x, filters)
    return x

# =========================
# BUILD MODEL
# =========================
def build_model():
    inputs = layers.Input((128,128,1))

    # Encoder
    c1 = res_block(inputs,32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = res_block(p1,64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = res_block(p2,128)
    p3 = layers.MaxPooling2D()(c3)

    # Bottleneck
    b = res_block(p3,256)

    # Decoder
    d1 = decoder_block(b, c3,128)
    d2 = decoder_block(d1,c2,64)
    d3 = decoder_block(d2,c1,32)

    outputs = layers.Conv2D(1,1,activation='sigmoid', dtype='float32')(d3)

    return Model(inputs, outputs)

model = build_model()
model.summary()

# =========================
# LOSSES (FIXED + STABLE)
# =========================
def dice_loss(y_true,y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    smooth = 1.
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    inter = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2*inter + smooth) / \
           (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def weighted_bce(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    eps = 1e-7
    y_pred = tf.clip_by_value(y_pred, eps, 1-eps)

    bce = -(y_true*tf.math.log(y_pred) +
            (1-y_true)*tf.math.log(1-y_pred))

    weights = y_true*10.0 + 1.0
    return tf.reduce_mean(bce * weights)

def dice_coef(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_pred = tf.cast(y_pred > 0.5, tf.float32)  # threshold

    smooth = 1.
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true_f * y_pred_f)

    return (2. * intersection + smooth) / \
           (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)
def soft_dice(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    smooth = 1.
    intersection = tf.reduce_sum(y_true * y_pred)

    return (2. * intersection + smooth) / \
           (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def final_loss(y_true,y_pred):
    return dice_loss(y_true,y_pred) + weighted_bce(y_true,y_pred)

# =========================
# COMPILE
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),
    loss=final_loss,
    metrics=[dice_coef,soft_dice]
)

# =========================
# TRAIN
# =========================
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(patience=2,factor=0.5),
    tf.keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)
]

history = model.fit(
    X, y,
    batch_size=4,
    epochs=30,
    validation_split=0.1,
    callbacks=callbacks
)

# =========================
# VALIDATION REPORT
# =========================
split = int(0.9*len(X))
X_val, y_val = X[split:], y[split:]

pred_val = model.predict(X_val, batch_size=32)

y_true = y_val.flatten()
y_pred = (pred_val.flatten() > 0.3).astype(int)

print("\n📊 CLASSIFICATION REPORT:\n")
print(classification_report(y_true, y_pred))

# =========================
# LOAD TEST
# =========================
def load_test():
    imgs, names = [], []

    for f in os.listdir(TEST_PATH):
        img = cv2.imread(os.path.join(TEST_PATH,f),0)
        img = cv2.resize(img,(128,128))/255.0
        img = apply_clahe(img)

        imgs.append(img)
        names.append(f)

    imgs = np.array(imgs)[...,np.newaxis]
    return imgs, names

X_test, names = load_test()

# =========================
# TTA (flip)
# =========================
pred1 = model.predict(X_test, batch_size=32)

flip = np.flip(X_test, axis=2)
pred2 = model.predict(flip, batch_size=32)
pred2 = np.flip(pred2, axis=2)

preds = (pred1 + pred2)/2

# =========================
# POST PROCESS
# =========================
def clean(mask):
    kernel = np.ones((5,5),np.uint8)
    mask = cv2.morphologyEx(mask.astype(np.uint8),
                           cv2.MORPH_CLOSE, kernel)
    return cv2.dilate(mask,kernel,1)

def rle(mask):
    pixels = mask.flatten()
    pixels = np.concatenate([[0],pixels,[0]])
    runs = np.where(pixels[1:]!=pixels[:-1])[0]+1
    runs[1::2] -= runs[::2]
    return ' '.join(map(str,runs))

pred_masks = [clean(p.squeeze()>0.3) for p in preds]
rles = [rle(m) for m in pred_masks]

df = pd.DataFrame({'img':names,'pixels':rles})
df.to_csv("submission.csv",index=False)

# =========================
# VISUALIZATION
# =========================
def show_preds(n=5):
    for i in range(n):
        plt.figure(figsize=(12,4))

        plt.subplot(1,3,1)
        plt.imshow(X_test[i].squeeze(),cmap='gray')
        plt.title("Original")

        plt.subplot(1,3,2)
        plt.imshow(preds[i].squeeze(),cmap='gray')
        plt.title("Raw")

        plt.subplot(1,3,3)
        plt.imshow(pred_masks[i],cmap='gray')
        plt.title("Final")

        plt.show()

show_preds()

print("\n🔥 ATTENTION RESUNET READY 🔥")

In [ ]:
sample_img_path = str(TRAIN_PATH / '1_1.tif') # Example image file
sample_image = preprocess_image(sample_img_path)

plt.figure(figsize=(6, 6))
plt.imshow(sample_image, cmap='gray')
plt.title(f"Sample Image (Processed) - {sample_image.shape[0]}x{sample_image.shape[1]}")
plt.axis('off')
plt.show()

# **Atention ResUNet + Deep Supervision ( Stable  Version)**

In [ ]:
# =========================
# IMPORTS
# =========================
import os, cv2, numpy as np, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from sklearn.metrics import classification_report

# =========================
# PATHS
# =========================
TRAIN_PATH = "/kaggle/input/competitions/ultrasound-nerve-segmentation/train"
TEST_PATH  = "/kaggle/input/competitions/ultrasound-nerve-segmentation/test"

# =========================
# DATA LOADING
# =========================
def load_data():
    images, masks = [], []
    files = os.listdir(TRAIN_PATH)
    image_files = [f for f in files if "_mask" not in f]

    for img_file in image_files:
        mask_file = img_file.replace(".tif", "_mask.tif")

        img = cv2.imread(os.path.join(TRAIN_PATH, img_file), 0)
        mask = cv2.imread(os.path.join(TRAIN_PATH, mask_file), 0)

        img = cv2.resize(img, (128,128)) / 255.0
        mask = cv2.resize(mask, (128,128))
        mask = (mask > 127).astype(np.float32)

        images.append(img)
        masks.append(mask)

        # AUGMENT
        images.append(np.fliplr(img))
        masks.append(np.fliplr(mask))

    X = np.array(images)[..., np.newaxis]
    y = np.array(masks)[..., np.newaxis]

    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

X, y = load_data()

# =========================
# METRICS + LOSS
# =========================
def dice_coef(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    smooth = 1.
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.*intersection + smooth) / \
           (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def bce_loss(y_true, y_pred):
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))

def final_loss(y_true, y_pred):
    return dice_loss(y_true, y_pred) + bce_loss(y_true, y_pred)

# =========================
# ATTENTION BLOCK
# =========================
def attention_block(x, g, filters):
    theta = layers.Conv2D(filters, 1)(x)
    phi   = layers.Conv2D(filters, 1)(g)

    add = layers.Add()([theta, phi])
    act = layers.Activation('relu')(add)
    psi = layers.Conv2D(1, 1, activation='sigmoid')(act)

    return layers.Multiply()([x, psi])

# =========================
# RESIDUAL BLOCK
# =========================
def res_block(x, filters):
    shortcut = layers.Conv2D(filters, 1, padding='same')(x)

    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

# =========================
# DECODER BLOCK
# =========================
def decoder_block(x, skip, filters):
    x = layers.UpSampling2D()(x)
    skip = attention_block(skip, x, filters)
    x = layers.Concatenate()([x, skip])
    x = res_block(x, filters)
    return x

# =========================
# BUILD MODEL
# =========================
def build_model():
    inputs = layers.Input((128,128,1))

    # Encoder
    c1 = res_block(inputs, 32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = res_block(p1, 64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = res_block(p2, 128)
    p3 = layers.MaxPooling2D()(c3)

    # Bottleneck
    b = res_block(p3, 256)

    # Decoder
    d1 = decoder_block(b, c3, 128)
    d2 = decoder_block(d1, c2, 64)
    d3 = decoder_block(d2, c1, 32)

    # =========================
    # DEEP SUPERVISION OUTPUTS
    # =========================
    out_main = layers.Conv2D(1,1,activation='sigmoid', name="main")(d3)

    out_d1 = layers.Conv2D(1,1,activation='sigmoid')(d1)
    out_d1 = layers.UpSampling2D(size=(4,4), name="aux1")(out_d1)

    out_d2 = layers.Conv2D(1,1,activation='sigmoid')(d2)
    out_d2 = layers.UpSampling2D(size=(2,2), name="aux2")(out_d2)

    return Model(inputs, [out_main, out_d1, out_d2])

model = build_model()

# =========================
# COMPILE
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss={
        "main": final_loss,
        "aux1": final_loss,
        "aux2": final_loss
    },
    loss_weights={
        "main": 1.0,
        "aux1": 0.3,
        "aux2": 0.3
    },
    metrics={"main": dice_coef}
)

# =========================
# TRAIN
# =========================
history = model.fit(
    X,
    {"main": y, "aux1": y, "aux2": y},
    batch_size=4,
    epochs=20,
    validation_split=0.1
)

# =========================
# VALIDATION REPORT
# =========================
val_split = int(0.9 * len(X))
X_val, y_val = X[val_split:], y[val_split:]

pred_val = model.predict(X_val)[0]

y_true_flat = y_val.flatten()
y_pred_flat = (pred_val.flatten() > 0.5).astype(int)

print("\n📊 CLASSIFICATION REPORT:\n")
print(classification_report(y_true_flat, y_pred_flat))

# =========================
# TEST LOADING
# =========================
def load_test():
    imgs, names = [], []
    for f in os.listdir(TEST_PATH):
        img = cv2.imread(os.path.join(TEST_PATH,f),0)
        img = cv2.resize(img,(128,128))/255.0
        imgs.append(img)
        names.append(f)
    return np.array(imgs)[...,np.newaxis], names

X_test, names = load_test()

# =========================
# PREDICT
# =========================
preds = model.predict(X_test)[0]

# =========================
# POST-PROCESS
# =========================
def clean(mask):
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

pred_masks = [clean((p.squeeze()>0.5)) for p in preds]

# =========================
# VISUALIZATION
# =========================
for i in range(5):
    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(X_test[i].squeeze(), cmap='gray')
    plt.title("Original")

    plt.subplot(1,3,2)
    plt.imshow(preds[i].squeeze(), cmap='gray')
    plt.title("Raw")

    plt.subplot(1,3,3)
    plt.imshow(pred_masks[i], cmap='gray')
    plt.title("Final")

    plt.show()

print("\n🔥 Attention ResUNet + Deep Supervision READY 🔥")
print(model.output_names)

CLAUDE

In [ ]:
# =============================================================================
# ULTRASOUND NERVE SEGMENTATION — BEST COMBINED ARCHITECTURE
# Papers: Lancia(2025) + Meiburger(2021) + Akkasaligar(2020) +
#         Sreeja(2025) + Gujarati(2024)
#
# Architecture: Attention U-Net + ResNet50 encoder (pretrained)
# Loss:         Dice + Focal (Gujarati et al.)
# Preprocessing: CLAHE + Median Filter + Normalization (Sreeja + Akkasaligar)
# Augmentation: Rotation/Flip/Elastic/GridDistortion (Sreeja + Gujarati)
# =============================================================================

# ── CELL 1: Install & Imports ─────────────────────────────────────────────────
import subprocess
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "segmentation-models-pytorch",
    "albumentations",
    "imagecodecs"   # ✅ ADD THIS
], check=True)




import os
import glob
import random
import numpy as np
import pandas as pd
import cv2
import tifffile
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import KFold

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── CELL 2: Paths & Config ────────────────────────────────────────────────────
BASE_PATH   = Path("/kaggle/input/competitions/ultrasound-nerve-segmentation")
TRAIN_PATH  = BASE_PATH / "train"
TEST_PATH   = BASE_PATH / "test"
SUBMIT_CSV  = BASE_PATH / "sample_submission.csv"

# Hyperparameters (tuned from Sreeja & Meiburger)
IMG_SIZE    = 256           # Meiburger: 256×256 crop
BATCH_SIZE  = 8             # Meiburger: batch_size=8
LR          = 2e-4          # Meiburger: Adam lr=0.0002
EPOCHS      = 50            # Meiburger & Gujarati: max 50 epochs
PATIENCE    = 10            # Meiburger: early stop at 25 epochs (reduced here)
N_FOLDS     = 5             # Meiburger: 5-fold cross-validation
ENCODER     = "resnet50"    # Meiburger: ResNet50 encoder
ENCODER_WEIGHTS = "imagenet"
THRESHOLD   = 0.5

# ── CELL 3: Preprocessing Functions (Akkasaligar + Sreeja) ───────────────────
def apply_clahe(img: np.ndarray) -> np.ndarray:
    """Contrast Limited Adaptive Histogram Equalization — Sreeja et al."""
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(img)

def apply_median_filter(img: np.ndarray, ksize: int = 3) -> np.ndarray:
    """Median filtering for speckle noise removal — Akkasaligar et al."""
    return cv2.medianBlur(img, ksize)

def preprocess_image(img_path: str) -> np.ndarray:
    """
    Full pipeline:
      1. Load TIFF as grayscale
      2. Median filter  (Akkasaligar)
      3. CLAHE           (Sreeja)
      4. Resize 256×256  (Meiburger)
      5. Normalise [0,1] (Lancia)
    Returns uint8 array for albumentations.
    """
    img = tifffile.imread(img_path)
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    img = apply_median_filter(img, ksize=3)
    img = apply_clahe(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    return img   # uint8, single channel

def preprocess_mask(mask_path: str) -> np.ndarray:
    """Load, resize, binarise mask — Lancia et al."""
    mask = tifffile.imread(mask_path)
    if mask.ndim == 3:
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 127).astype(np.uint8)   # binary: 0 or 1
    return mask

# ── CELL 4: Augmentation Pipelines (Sreeja + Gujarati) ───────────────────────
def get_train_transforms():
    """
    Aggressive augmentation:
    - Flip / Rotate / Shift-Scale-Rotate
    - Elastic / Grid distortion (nerve shape variability)
    - Brightness / Contrast
    """
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15,
                           rotate_limit=20, p=0.6,
                           border_mode=cv2.BORDER_REFLECT),
        A.ElasticTransform(alpha=120, sigma=120 * 0.05,
                           alpha_affine=120 * 0.03, p=0.4),
        A.GridDistortion(p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2,
                                   contrast_limit=0.2, p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.Normalize(mean=[0.485], std=[0.229]),   # ImageNet-style for 1-ch
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=[0.485], std=[0.229]),
        ToTensorV2(),
    ])

# ── CELL 5: Dataset ───────────────────────────────────────────────────────────
def collect_train_pairs(train_dir: Path):
    """Return list of (image_path, mask_path) tuples."""
    mask_files = sorted(train_dir.glob("*_mask.tif"))
    pairs = []
    for mf in mask_files:
        img_name = mf.name.replace("_mask.tif", ".tif")
        img_path = train_dir / img_name
        if img_path.exists():
            pairs.append((str(img_path), str(mf)))
    print(f"Found {len(pairs)} image-mask pairs.")
    return pairs

class NerveDataset(Dataset):
    def __init__(self, pairs, transforms=None):
        self.pairs = pairs
        self.transforms = transforms

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img  = preprocess_image(img_path)     # (H, W) uint8
        mask = preprocess_mask(mask_path)     # (H, W) uint8

        # Expand to 3-channel for ResNet50 encoder (ImageNet pretrained)
        img_3ch = np.stack([img, img, img], axis=-1)   # (H, W, 3)

        if self.transforms:
            augmented = self.transforms(image=img_3ch, mask=mask)
            img_3ch = augmented["image"]     # (3, H, W) tensor
            mask    = augmented["mask"].unsqueeze(0).float()  # (1, H, W)

        return img_3ch, mask

class TestDataset(Dataset):
    def __init__(self, test_dir: Path, transforms=None):
        self.files = sorted(test_dir.glob("*.tif"))
        self.transforms = transforms

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fpath = self.files[idx]
        img   = preprocess_image(str(fpath))
        img_3ch = np.stack([img, img, img], axis=-1)
        if self.transforms:
            augmented = self.transforms(image=img_3ch)
            img_3ch   = augmented["image"]
        return img_3ch, fpath.stem   # tensor, filename stem (for submission)

# ── CELL 6: Combined Loss — Dice + Focal (Gujarati et al.) ───────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred   = torch.sigmoid(pred)
        pred   = pred.view(-1)
        target = target.view(-1)
        intersection = (pred * target).sum()
        return 1 - (2. * intersection + self.smooth) / \
                   (pred.sum() + target.sum() + self.smooth)

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.8, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        bce  = nn.functional.binary_cross_entropy_with_logits(pred, target,
                                                               reduction='none')
        p_t  = torch.exp(-bce)
        loss = self.alpha * ((1 - p_t) ** self.gamma) * bce
        return loss.mean()

class CombinedLoss(nn.Module):
    """Dice + Focal — Gujarati et al. Eq.(6)"""
    def __init__(self, dice_w=0.5, focal_w=0.5):
        super().__init__()
        self.dice  = DiceLoss()
        self.focal = FocalLoss()
        self.dw    = dice_w
        self.fw    = focal_w

    def forward(self, pred, target):
        return self.dw * self.dice(pred, target) + \
               self.fw * self.focal(pred, target)

# ── CELL 7: Metrics ───────────────────────────────────────────────────────────
def dice_score(pred: torch.Tensor, target: torch.Tensor,
               threshold: float = 0.5, smooth: float = 1.0) -> float:
    pred   = (torch.sigmoid(pred) > threshold).float()
    pred   = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return ((2. * intersection + smooth) /
            (pred.sum() + target.sum() + smooth)).item()

def iou_score(pred: torch.Tensor, target: torch.Tensor,
              threshold: float = 0.5, smooth: float = 1.0) -> float:
    pred   = (torch.sigmoid(pred) > threshold).float()
    pred   = pred.view(-1).long()
    target = target.view(-1).long()
    intersection = (pred & target).float().sum()
    union        = (pred | target).float().sum()
    return ((intersection + smooth) / (union + smooth)).item()

# ── CELL 8: Model Builder ─────────────────────────────────────────────────────
def build_model() -> nn.Module:
    """
    Attention U-Net + ResNet50 (ImageNet pretrained)
    — Combines Meiburger's UNet+ResNet50 with Sreeja's attention mechanism
    """
    model = smp.Unet(
        encoder_name    = ENCODER,
        encoder_weights = ENCODER_WEIGHTS,
        in_channels     = 3,
        classes         = 1,
        decoder_attention_type = "scse",   # Squeeze-and-Excitation attention
    )
    return model.to(DEVICE)

# ── CELL 9: Train / Validation Loops ─────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_dice = 0.0, 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_dice += dice_score(preds.detach(), masks.detach())
    n = len(loader)
    return total_loss / n, total_dice / n

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, total_dice, total_iou = 0.0, 0.0, 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = model(imgs)
        loss  = criterion(preds, masks)
        total_loss += loss.item()
        total_dice += dice_score(preds, masks)
        total_iou  += iou_score(preds, masks)
    n = len(loader)
    return total_loss / n, total_dice / n, total_iou / n

# ── CELL 10: Full Training Pipeline with 5-Fold CV ───────────────────────────
def run_training(pairs):
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(pairs)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold+1}/{N_FOLDS}")
        print(f"{'='*60}")

        train_pairs = [pairs[i] for i in train_idx]
        val_pairs   = [pairs[i] for i in val_idx]

        train_ds = NerveDataset(train_pairs, transforms=get_train_transforms())
        val_ds   = NerveDataset(val_pairs,   transforms=get_val_transforms())

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                                  shuffle=True,  num_workers=2, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                                  shuffle=False, num_workers=2, pin_memory=True)

        model     = build_model()
        criterion = CombinedLoss(dice_w=0.5, focal_w=0.5)
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        # ReduceLROnPlateau — Meiburger: lr decreases by factor 10 every 10
        # epochs without improvement
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.1, patience=10)

        best_dice   = 0.0
        no_improve  = 0
        history     = {"train_loss": [], "val_loss": [],
                       "train_dice": [], "val_dice": [], "val_iou": []}

        for epoch in range(1, EPOCHS + 1):
            tr_loss, tr_dice = train_one_epoch(model, train_loader,
                                               optimizer, criterion)
            vl_loss, vl_dice, vl_iou = validate(model, val_loader, criterion)
            scheduler.step(vl_dice)

            history["train_loss"].append(tr_loss)
            history["val_loss"].append(vl_loss)
            history["train_dice"].append(tr_dice)
            history["val_dice"].append(vl_dice)
            history["val_iou"].append(vl_iou)

            print(f"Epoch [{epoch:3d}/{EPOCHS}] "
                  f"| Train Loss: {tr_loss:.4f}  Dice: {tr_dice:.4f} "
                  f"| Val Loss: {vl_loss:.4f}  Dice: {vl_dice:.4f}  "
                  f"IoU: {vl_iou:.4f}")

            if vl_dice > best_dice:
                best_dice = vl_dice
                no_improve = 0
                torch.save(model.state_dict(),
                           f"/kaggle/working/best_model_fold{fold+1}.pth")
                print(f"  ✓ Saved best model (Dice={best_dice:.4f})")
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    print(f"  Early stopping at epoch {epoch}")
                    break

        fold_results.append({"fold": fold+1, "best_dice": best_dice})
        plot_fold_history(history, fold+1)

    print("\n" + "="*60)
    print("CROSS-VALIDATION RESULTS")
    for r in fold_results:
        print(f"  Fold {r['fold']}: Best Dice = {r['best_dice']:.4f}")
    mean_dice = np.mean([r["best_dice"] for r in fold_results])
    print(f"  Mean Dice = {mean_dice:.4f}")
    print("="*60)
    return fold_results

def plot_fold_history(history, fold_num):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="Train Loss")
    axes[0].plot(history["val_loss"],   label="Val Loss")
    axes[0].set_title(f"Fold {fold_num} — Loss"); axes[0].legend()
    axes[1].plot(history["train_dice"], label="Train Dice")
    axes[1].plot(history["val_dice"],   label="Val Dice")
    axes[1].plot(history["val_iou"],    label="Val IoU")
    axes[1].set_title(f"Fold {fold_num} — Metrics"); axes[1].legend()
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/fold{fold_num}_history.png", dpi=100)
    plt.show()

# ── CELL 11: RLE Encoding for Submission ─────────────────────────────────────
def rle_encode(mask: np.ndarray) -> str:
    """Run-Length Encoding for Kaggle submission."""
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs) if len(runs) else ''

# ── CELL 12: Ensemble Inference (average across folds) ───────────────────────
@torch.no_grad()
def predict_ensemble(test_loader, n_folds=N_FOLDS):
    """
    Load each fold's best weights and average predictions.
    Ensemble = more robust than single-fold (Meiburger: unbiased estimation).
    """
    all_preds = {}   # stem -> accumulated probability map

    for fold in range(1, n_folds + 1):
        weight_path = f"/kaggle/working/best_model_fold{fold}.pth"
        if not os.path.exists(weight_path):
            print(f"  Fold {fold} weights not found, skipping.")
            continue
        model = build_model()
        model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
        model.eval()
        print(f"  Running inference — Fold {fold}")

        for imgs, stems in test_loader:
            imgs  = imgs.to(DEVICE)
            preds = torch.sigmoid(model(imgs)).cpu().numpy()  # (B,1,H,W)
            for pred, stem in zip(preds, stems):
                all_preds.setdefault(stem, np.zeros((IMG_SIZE, IMG_SIZE)))
                all_preds[stem] += pred[0]   # accumulate across folds

    # Average and threshold
    results = {}
    for stem, prob_map in all_preds.items():
        avg_map = prob_map / n_folds
        results[stem] = (avg_map > THRESHOLD).astype(np.uint8)

    return results

# ── CELL 13: Generate Submission CSV ─────────────────────────────────────────
def generate_submission(results: dict, sample_csv_path: Path):
    sample = pd.read_csv(sample_csv_path)
    rle_dict = {stem: rle_encode(mask) for stem, mask in results.items()}

    # Map img integer id to rle
    sample["img_id"] = sample["img"].astype(str)
    sample["pixels"] = sample["img_id"].map(rle_dict).fillna('')
    submission = sample[["img", "pixels"]]
    out_path   = "/kaggle/working/submission.csv"
    submission.to_csv(out_path, index=False)
    print(f"Submission saved → {out_path}")
    print(submission.head(10))
    return submission

# ── CELL 14: Visualise Predictions (QA — Lancia et al. visual inspection) ────
@torch.no_grad()
def visualise_samples(pairs, n=5, fold=1):
    """Show image / ground-truth mask / predicted mask for quality assurance."""
    model = build_model()
    model.load_state_dict(
        torch.load(f"/kaggle/working/best_model_fold{fold}.pth",
                   map_location=DEVICE))
    model.eval()

    transforms = get_val_transforms()
    sample_pairs = random.sample(pairs, min(n, len(pairs)))

    fig, axes = plt.subplots(n, 3, figsize=(12, n * 4))
    axes[0][0].set_title("Ultrasound Image",  fontsize=13)
    axes[0][1].set_title("Ground Truth Mask", fontsize=13)
    axes[0][2].set_title("Predicted Mask",    fontsize=13)

    for row, (img_path, mask_path) in enumerate(sample_pairs):
        img_raw  = preprocess_image(img_path)
        mask_gt  = preprocess_mask(mask_path)

        img_3ch  = np.stack([img_raw, img_raw, img_raw], axis=-1)
        aug      = transforms(image=img_3ch)
        tensor   = aug["image"].unsqueeze(0).to(DEVICE)

        pred     = torch.sigmoid(model(tensor)).squeeze().cpu().numpy()
        pred_bin = (pred > THRESHOLD).astype(np.uint8)

        d = dice_score(
            torch.sigmoid(model(tensor)).detach(),
            torch.tensor(mask_gt).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
        )

        axes[row][0].imshow(img_raw,  cmap="gray")
        axes[row][0].axis("off")
        axes[row][1].imshow(mask_gt,  cmap="gray")
        axes[row][1].axis("off")
        axes[row][2].imshow(pred_bin, cmap="gray")
        axes[row][2].set_xlabel(f"Dice={d:.3f}", fontsize=11)
        axes[row][2].axis("off")

    plt.tight_layout()
    plt.savefig("/kaggle/working/sample_predictions.png", dpi=120)
    plt.show()

# ── CELL 15: MAIN ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # ── 1. Collect data ──────────────────────────────────────────────────────
    pairs = collect_train_pairs(TRAIN_PATH)

    # ── 2. Quick dataset sanity check (Lancia — visual inspection) ───────────
    print("\nSanity check — first pair:")
    from PIL import Image

    def preprocess_image(img_path):
        img = np.array(Image.open(img_path))
        if img.ndim == 3:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        img = apply_median_filter(img, ksize=3)
        img = apply_clahe(img)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        return img
    img_sample  = preprocess_image(pairs[0][0])
    mask_sample = preprocess_mask(pairs[0][1])
    print(f"  Image shape: {img_sample.shape}, dtype: {img_sample.dtype}")
    print(f"  Mask shape:  {mask_sample.shape}, unique: {np.unique(mask_sample)}")

    # ── 3. Train with 5-fold CV ───────────────────────────────────────────────
    fold_results = run_training(pairs)

    # ── 4. Visualise predictions ──────────────────────────────────────────────
    visualise_samples(pairs, n=5, fold=1)

    # ── 5. Ensemble inference on test set ────────────────────────────────────
    print("\nRunning ensemble inference on test set …")
    test_ds     = TestDataset(TEST_PATH, transforms=get_val_transforms())
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=2)
    results     = predict_ensemble(test_loader)

    # ── 6. Generate submission ────────────────────────────────────────────────
    if SUBMIT_CSV.exists():
        submission = generate_submission(results, SUBMIT_CSV)
    else:
        print("sample_submission.csv not found — skipping submission generation.")

    print("\nDone.")

In [ ]:
!pip install imagecodecs --no-cache-dir

In [ ]:
import imagecodecs
print("imagecodecs installed ✅")

In [ ]:
!pip install tifffile

In [ ]:
import tifffile
print("tifffile installed successfully!")

## Report Section: Data Overview and Preprocessing

This section demonstrates the input data used for training and the initial preprocessing applied. We work with ultrasound images and their corresponding nerve masks. The preprocessing involves resizing and potentially techniques like CLAHE for contrast enhancement.

In [ ]:
import matplotlib.pyplot as plt

# Assuming train_dataset, img, and mask variables are available from previous cells.
# If not, you might need to re-run relevant data loading and preprocessing cells.

# Display a sample image and its mask from the training dataset
if 'train_dataset' in locals():
    sample_img, sample_mask = train_dataset[0]

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(sample_img.cpu().squeeze(), cmap='gray')
    plt.title("Sample Ultrasound Image (Processed)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(sample_mask.cpu().squeeze(), cmap='gray')
    plt.title("Corresponding Nerve Mask (Ground Truth)")
    plt.axis('off')
    plt.show()
else:
    print("train_dataset not found. Please ensure data loading and dataset creation cells have been run.")

## Report Section: Model Training and Performance

Here, we visualize the training progress of our Attention U-Net model. The plots below show how the loss and Dice coefficient (a common metric for segmentation tasks) evolve over epochs for both the training and validation sets. This helps in understanding if the model is learning effectively and generalizing well to unseen data.

In [ ]:
import matplotlib.pyplot as plt

# Assuming `history` variable is available from model.fit()
# (e.g., from cell szxhRhwXIdc5 or XnB8Dr4VIdc4 for a TensorFlow model)

if 'history' in locals() and history is not None:
    # Plot training & validation loss values
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss Over Epochs')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(loc='upper right')

    # Plot training & validation dice_coef values
    plt.subplot(1, 2, 2)
    plt.plot(history.history['dice_coef'], label='Train Dice Coef')
    plt.plot(history.history['val_dice_coef'], label='Validation Dice Coef')
    plt.title('Model Dice Coefficient Over Epochs')
    plt.ylabel('Dice Coefficient')
    plt.xlabel('Epoch')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()
else:
    print("Training history not found. Please ensure a model training cell that stores history has been executed.")

## Report Section: Model Predictions and Qualitative Evaluation

This section provides visual examples of the model's predictions on unseen test data. Each example shows the original ultrasound image, the raw (unthresholded) prediction from the model, and the final post-processed binary mask. This allows for a qualitative assessment of how well the model identifies the nerve structures.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2 # Make sure cv2 is imported if `clean` function uses it

# Assuming X_test, preds, and the `clean` function are available from previous cells
# (e.g., from the final pipeline in cell szxhRhwXIdc5)

if 'X_test' in locals() and 'preds' in locals() and 'clean' in locals():
    # Function to display predictions (from szxhRhwXIdc5)
    def show_preds_report(X_test_report, preds_report, n=5):
        for i in range(n):
            plt.figure(figsize=(12,4))

            plt.subplot(1,3,1)
            plt.imshow(X_test_report[i].squeeze(),cmap='gray')
            plt.title("Original Image")
            plt.axis('off')

            plt.subplot(1,3,2)
            plt.imshow(preds_report[i].squeeze(),cmap='gray')
            plt.title("Raw Prediction (Probability Map)")
            plt.axis('off')

            plt.subplot(1,3,3)
            # Use the `clean` function and a threshold (e.g., 0.3 or 0.25)
            final_mask = clean(preds_report[i].squeeze()>0.25)
            plt.imshow(final_mask,cmap='gray')
            plt.title("Final Cleaned Mask")
            plt.axis('off')

            plt.show()

    # Function to overlay prediction (from szxhRhwXIdc5)
    def overlay_prediction_report(img, pred, threshold=0.3):
        mask = (pred.squeeze() > threshold).astype(np.uint8)
        img = (img.squeeze()*255).astype(np.uint8)
        img_color = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        img_color[:,:,1] = np.maximum(img_color[:,:,1], mask*255) # Green overlay
        return img_color

    print("Displaying sample predictions:")
    show_preds_report(X_test, preds, n=3)

    print("\nDisplaying overlay predictions:")
    for i in range(3):
        overlay = overlay_prediction_report(X_test[i], preds[i])
        plt.figure(figsize=(6,6))
        plt.imshow(overlay)
        plt.title("Original with Mask Overlay")
        plt.axis('off')
        plt.show()
else:
    print("Variables for test data or prediction functions (X_test, preds, clean) not found. Please ensure the full pipeline has been executed.")